# Lecture 1: Introduction to Time Series Analysis
- 1.1 Time Series Examples
- 1.2 Objectives of Time Series Analysis
- 1.3 Simple Time Series Models
- 1.4 Stationary Models and the Autocorrelation Function
- 1.5 Estimation and Elimination of Trend and Seasonality
- 1.6 Testing the Noise Sequence


In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 12
print("Libraries loaded successfully!")


## 1.1 Time Series Examples

A **time series** is a collection of observations recorded at specific time points:
$$\{x_t, t \in T_0\}$$

Let us reproduce the key examples from the textbook using Python.


In [ ]:
# ── Example 1: Australian Red Wine Sales (simulated) ──
np.random.seed(42)
n = 142  # Jan 1980 – Oct 1991
t = np.arange(n)

# Trend + seasonality + noise
trend = 0.01 * t
seasonal = 0.3 * np.sin(2 * np.pi * t / 12) + 0.1 * np.cos(2 * np.pi * t / 12)
noise = np.random.normal(0, 0.08, n)
wine = 1.0 + trend + seasonal + noise

dates = pd.date_range('1980-01', periods=n, freq='ME')

fig, axes = plt.subplots(3, 1, figsize=(13, 10))

# Wine sales
axes[0].plot(dates, wine, color='darkred', lw=1.5)
axes[0].set_title('Australian Red Wine Sales (Jan 1980 – Oct 1991)', fontsize=13)
axes[0].set_ylabel('Sales (thousands kL)')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Accidental deaths (USA, 1973-1978)
n2 = 72
t2 = np.arange(n2)
deaths = (9 + 0.5*np.sin(2*np.pi*t2/12) + np.random.normal(0, 0.3, n2))
dates2 = pd.date_range('1973-01', periods=n2, freq='ME')
axes[1].plot(dates2, deaths, color='steelblue', lw=1.5)
axes[1].set_title('US Accidental Deaths (1973–1978)', fontsize=13)
axes[1].set_ylabel('Deaths (thousands)')

# US Population (1790-1990)
years = np.arange(1790, 2000, 10)
pop = 3.9 * np.exp(0.02 * (years - 1790)) + np.random.normal(0, 1, len(years))
pop = np.clip(pop, 0, None)
axes[2].plot(years, pop, 'o-', color='green', lw=1.5, markersize=4)
axes[2].set_title('US Population (1790–1990)', fontsize=13)
axes[2].set_ylabel('Population (millions)')

plt.tight_layout()
plt.show()
print("Three different types of time series:")
print("  1. Trend + Seasonality (wine sales)")
print("  2. Seasonality only (deaths)")
print("  3. Exponential trend (population)")


## 1.3 Simple Time Series Models

### 1.3.1 Zero-Mean Models

**White Noise:** $\{Z_t\} \sim WN(0, \sigma^2)$, i.e. $E[Z_t]=0$, $\text{Cov}(Z_t, Z_s)=0$ ($t\neq s$).

**Moving Average (MA):** $X_t = \frac{1}{3}(Z_{t-1} + Z_t + Z_{t+1})$

**Autoregressive (AR):** $X_t = \phi X_{t-1} + Z_t$


In [ ]:
# ── Zero-Mean Models ──
np.random.seed(0)
n = 200
Z = np.random.normal(0, 1, n + 2)

# White Noise
WN = Z[1:-1]

# Moving Average MA(1): 3-point
MA = np.array([(Z[t-1] + Z[t] + Z[t+1]) / 3 for t in range(1, n+1)])

# AR(1): X_t = 0.9 * X_{t-1} + Z_t
AR = np.zeros(n)
AR[0] = Z[1]
phi = 0.9
for t in range(1, n):
    AR[t] = phi * AR[t-1] + Z[t+1]

fig, axes = plt.subplots(3, 1, figsize=(13, 9))
models = [('White Noise $Z_t$', WN, 'gray'),
          ('Moving Average $X_t = (Z_{t-1}+Z_t+Z_{t+1})/3$', MA, 'steelblue'),
          (r'AR(1): $X_t = 0.9 X_{t-1} + Z_t$', AR, 'darkorange')]

for ax, (title, data, color) in zip(axes, models):
    ax.plot(data, color=color, lw=1.2)
    ax.axhline(0, color='black', lw=0.5, ls='--')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Time')

plt.tight_layout()
plt.show()

print(f"White Noise     — Mean: {WN.mean():.3f}, Std: {WN.std():.3f}")
print(f"Moving Average  — Mean: {MA.mean():.3f}, Std: {MA.std():.3f}")
print(f"AR(1) phi=0.9   — Mean: {AR.mean():.3f}, Std: {AR.std():.3f}")


### 1.3.2 Models with Trend and Seasonality

General model:
$$X_t = m_t + s_t + Y_t$$

where:
- $m_t$: deterministic trend component
- $s_t$: seasonal component ($\sum_{j=1}^{d} s_j = 0$)
- $Y_t$: stationary noise process


In [ ]:
# ── Trend + Seasonality Decomposition ──
np.random.seed(7)
n = 96  # 8 years, monthly data
t = np.arange(1, n + 1)

# Components
trend = 0.05 * t          # Linear trend
seasonal = 2 * np.sin(2 * np.pi * t / 12)   # Monthly seasonality
noise = np.random.normal(0, 0.5, n)
X = trend + seasonal + noise

fig, axes = plt.subplots(4, 1, figsize=(13, 12), sharex=True)
axes[0].plot(t, X, 'k', lw=1.2); axes[0].set_title('Observed Series $X_t$')
axes[1].plot(t, trend, 'r', lw=1.5); axes[1].set_title('Trend $m_t = 0.05t$')
axes[2].plot(t, seasonal, 'b', lw=1.5); axes[2].set_title('Seasonality $s_t$')
axes[3].plot(t, noise, 'g', lw=1.2); axes[3].set_title('Noise $Y_t$')
for ax in axes:
    ax.axhline(0, color='gray', lw=0.5, ls='--')
    ax.set_ylabel('Value')
axes[-1].set_xlabel('Time (months)')
plt.tight_layout()
plt.show()


## 1.4 Stationary Models and the Autocorrelation Function

A process $\{X_t\}$ is **(weakly) stationary** if:
1. $E[X_t] = \mu$ (constant mean)
2. $\gamma(t+h, t) = \text{Cov}(X_{t+h}, X_t)$ depends only on $h$

**Autocovariance function:** $\gamma(h) = \text{Cov}(X_{t+h}, X_t)$

**Autocorrelation function (ACF):** $\rho(h) = \frac{\gamma(h)}{\gamma(0)}$

**Sample ACF:**
$$\hat{\rho}(h) = \frac{\sum_{t=1}^{n-h}(X_{t+h} - \bar{X})(X_t - \bar{X})}{\sum_{t=1}^{n}(X_t - \bar{X})^2}$$


In [ ]:
# ── ACF Computation and Interpretation ──
np.random.seed(42)
n = 300

# Different processes
WN = np.random.normal(0, 1, n)

# AR(1) phi=0.8
AR1_pos = np.zeros(n)
AR1_pos[0] = WN[0]
for t in range(1, n):
    AR1_pos[t] = 0.8 * AR1_pos[t-1] + np.random.normal(0, 1)

# AR(1) phi=-0.8
AR1_neg = np.zeros(n)
AR1_neg[0] = WN[0]
for t in range(1, n):
    AR1_neg[t] = -0.8 * AR1_neg[t-1] + np.random.normal(0, 1)

fig, axes = plt.subplots(3, 2, figsize=(14, 10))
series = [('White Noise', WN), ('AR(1) phi=+0.8', AR1_pos), ('AR(1) phi=-0.8', AR1_neg)]

for i, (name, s) in enumerate(series):
    axes[i, 0].plot(s[:100], lw=1, color='steelblue')
    axes[i, 0].set_title(f'{name} — Time Series')
    axes[i, 0].axhline(0, color='gray', ls='--', lw=0.5)
    
    plot_acf(s, lags=20, ax=axes[i, 1], color='steelblue', title=f'{name} — ACF')

plt.tight_layout()
plt.show()

print("Theoretical verification:")
print(f"AR(1) phi=0.8  → Theoretical rho(1) = {0.8:.3f},  Computed rho(1) ≈ {acf(AR1_pos, nlags=1)[1]:.3f}")
print(f"AR(1) phi=-0.8 → Theoretical rho(1) = {-0.8:.3f}, Computed rho(1) ≈ {acf(AR1_neg, nlags=1)[1]:.3f}")


## 1.5 Estimation and Elimination of Trend and Seasonality

### 1.5.1 Trend Estimation without Seasonality

**Moving Average Filter:**
$$\hat{m}_t = \frac{1}{2q+1}\sum_{j=-q}^{q} X_{t+j}$$

**Polynomial Regression:** $m_t = a_0 + a_1 t + a_2 t^2 + \ldots$


In [ ]:
# ── Trend Estimation: Moving Average vs Polynomial Regression ──
np.random.seed(10)
n = 100
t = np.arange(1, n+1)

# True trend: quadratic
true_trend = 0.002 * t**2 - 0.1 * t + 5
X = true_trend + np.random.normal(0, 0.8, n)

# 1) Moving Average (q=5, window=11)
q = 5
MA_trend = np.full(n, np.nan)
for i in range(q, n-q):
    MA_trend[i] = X[i-q:i+q+1].mean()

# 2) Polynomial Regression (degree=2)
coeffs = np.polyfit(t, X, 2)
poly_trend = np.polyval(coeffs, t)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(t, X, 'lightgray', lw=1, label='Observations')
axes[0].plot(t, true_trend, 'k--', lw=2, label='True Trend')
axes[0].plot(t, MA_trend, 'r', lw=2, label=f'MA (q={q})')
axes[0].plot(t, poly_trend, 'b', lw=2, label='Polynomial (degree 2)')
axes[0].legend(); axes[0].set_title('Trend Estimation Methods')

# Residuals
resid_poly = X - poly_trend
axes[1].plot(t, resid_poly, color='steelblue', lw=1)
axes[1].axhline(0, color='red', ls='--')
axes[1].set_title('Residuals After Trend Removal')
axes[1].set_xlabel('Time')

plt.tight_layout()
plt.show()

print(f"Polynomial coefficients: a2={coeffs[0]:.4f}, a1={coeffs[1]:.4f}, a0={coeffs[2]:.4f}")
print(f"True coefficients:        a2=0.0020,      a1=-0.1000,     a0=5.0000")


### 1.5.2 Joint Estimation and Elimination of Trend and Seasonality

Trend is removed by first applying seasonal differencing:
$$\nabla_d X_t = X_t - X_{t-d}$$

Followed by a second differencing for the trend:
$$\nabla \nabla_d X_t = (1-B)(1-B^d)X_t$$


In [ ]:
# ── Trend + Seasonality Decomposition: Classical Method ──
from statsmodels.tsa.seasonal import seasonal_decompose

np.random.seed(5)
n = 120  # 10 years, monthly
t_idx = np.arange(1, n+1)

trend_comp = 0.05 * t_idx
seas_comp = 1.5 * np.sin(2 * np.pi * t_idx / 12) + 0.3 * np.sin(4 * np.pi * t_idx / 12)
noise_comp = np.random.normal(0, 0.25, n)
X = trend_comp + seas_comp + noise_comp

dates = pd.date_range('2013-01', periods=n, freq='ME')
ts = pd.Series(X, index=dates)

# Seasonal decomposition
result = seasonal_decompose(ts, model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
result.observed.plot(ax=axes[0], color='k', lw=1.2, title='Observed')
result.trend.plot(ax=axes[1], color='r', lw=1.5, title='Trend')
result.seasonal.plot(ax=axes[2], color='b', lw=1.5, title='Seasonality')
result.resid.plot(ax=axes[3], color='g', lw=1.2, title='Residual')
for ax in axes:
    ax.axhline(0, color='gray', lw=0.5, ls='--')
plt.tight_layout()
plt.show()

# Eliminating seasonality via differencing
diff12 = ts.diff(12).dropna()
diff1_12 = diff12.diff(1).dropna()

fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=False)
ts.plot(ax=axes[0], title='Original Series $X_t$', color='k')
diff12.plot(ax=axes[1], title='Seasonal Difference $\\nabla_{12} X_t$', color='blue')
diff1_12.plot(ax=axes[2], title='Seasonal + Trend Difference $\\nabla \\nabla_{12} X_t$', color='green')
plt.tight_layout()
plt.show()


## 1.6 Testing the Noise Sequence

To test the independence of residuals:
- **Portmanteau (Ljung-Box) test:** $H_0$: No autocorrelation at the first $h$ lags
$$Q = n(n+2) \sum_{j=1}^{h} \frac{\hat{\rho}^2(j)}{n-j} \sim \chi^2(h)$$


In [ ]:
# ── Ljung-Box Test ──
from statsmodels.stats.diagnostic import acorr_ljungbox

np.random.seed(99)

# Test 1: True White Noise
WN_test = np.random.normal(0, 1, 200)

# Test 2: AR(1) residuals (misspecified model)
AR_test = np.zeros(200)
for t in range(1, 200):
    AR_test[t] = 0.7 * AR_test[t-1] + np.random.normal(0, 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for i, (name, s, color) in enumerate([('White Noise', WN_test, 'steelblue'), 
                                        ('AR(1) Residuals', AR_test, 'crimson')]):
    axes[i, 0].plot(s, lw=0.8, color=color)
    axes[i, 0].axhline(0, color='k', ls='--', lw=0.5)
    axes[i, 0].set_title(f'{name} — Time Series')
    plot_acf(s, lags=20, ax=axes[i, 1], color=color, title=f'{name} — ACF')

plt.tight_layout()
plt.show()

print("=" * 60)
print(f"{'Series':<22} {'Lag':>5} {'LB Statistic':>15} {'p-value':>12}")
print("=" * 60)
for name, s in [('White Noise', WN_test), ('AR(1) Residuals', AR_test)]:
    lb = acorr_ljungbox(s, lags=[10, 20], return_df=True)
    for lag, row in lb.iterrows():
        print(f"{name:<22} {int(lag):>5} {row['lb_stat']:>15.3f} {row['lb_pvalue']:>12.4f}")
    print()

print("p < 0.05 => H0 rejected => Residuals are NOT independent (inadequate model)")
print("p > 0.05 => H0 not rejected => Residuals appear independent (adequate model)")


## Summary

| Concept | Description |
|---------|-------------|
| **Time Series** | Sequence of observations indexed by time |
| **Trend** | Long-term direction ($m_t$) |
| **Seasonality** | Periodically repeating pattern ($s_t$) |
| **Stationarity** | Constant mean and covariance structure |
| **ACF** | Measure of lagged correlation $\rho(h)$ |
| **Ljung-Box** | Independence test for residuals |

